In [0]:
# Silver Agents

from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_agents"
silver_table = f"{CATALOG}.{SILVER_SCHEMA}.silver_agents"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

agents_bronze = spark.table(bronze_table)

agents_prepared = (
    agents_bronze
    .withColumn("agent_id", F.trim(F.col("agent_id")))
    .withColumn("agent_name", F.trim(F.col("agent_name")))
    .withColumn("region", F.lower(F.trim(F.col("region"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("bundesland", F.trim(F.col("bundesland")))
    .withColumn("commission_rate", F.col("commission_rate").cast("double"))
    .withColumn("active_flag", F.col("active_flag").cast("boolean"))
)

valid_agents = (
    agents_prepared
    .filter(
        F.col("agent_id").isNotNull()
        & F.col("commission_rate").isNotNull()
        & (F.col("commission_rate") >= 0)
        & (F.col("commission_rate") <= 1)
        & F.col("active_flag").isNotNull()
    )
    .dropDuplicates(["agent_id"])
)

valid_agents.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

print("Bronze agents:", agents_bronze.count())
print("Silver agents:", spark.table(silver_table).count())